# 你的第一个简单工具

在上一节课中，我们介绍了工具使用的工作流程。现在是时候开始实现一个简单的工具使用示例了。回顾一下，工具使用过程最多有4个步骤：

1. **为 Claude 提供工具和用户提示：**（API 请求）
    * 定义你希望 Claude 访问的工具集，包括它们的名称、描述和输入模式。
    * 提供一个可能需要使用一个或多个工具来回答的用户提示。

2. **Claude 使用工具：**（API 响应）
    * Claude 评估用户提示，决定可用的工具是否有助于回答用户的查询或任务。如果有，它还决定使用哪个工具以及使用什么输入。
    * Claude 输出一个正确格式化的工具使用请求。
    * API 响应将有一个 `stop_reason` 为 `tool_use`，表示 Claude 想要使用外部工具。

3. **提取工具输入、运行代码并返回结果：**（API 请求）
    * 在客户端，你应该从 Claude 的工具使用请求中提取工具名称和输入。
    * 在客户端运行实际的工具代码。
    * 通过继续对话并包含一个 `tool_result` 内容块的新用户消息，将结果返回给 Claude。

4. **Claude 使用工具结果来形成响应：**（API 响应）
    * 收到工具结果后，Claude 将使用这些信息来形成对原始用户提示的最终响应。

我们将从一个简单的演示开始，这个演示只需要与 Claude "对话"一次（别担心，我们很快就会看到更令人兴奋的例子！）。这意味着我们暂时不涉及第4步。我们将要求 Claude 回答一个问题，Claude 将请求使用工具来回答它，然后我们将提取工具输入、运行代码并返回结果。

当今的大型语言模型在数学运算方面存在困难，以下代码可以证明这一点。

我们要求 Claude "将 1984135 乘以 9343116"：

In [22]:
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic()

# A relatively simple math problem
response = client.messages.create(
    model="claude-3-haiku-20240307",
    messages=[{"role": "user", "content":"Multiply 1984135 by 9343116. Only respond with the result"}],
    max_tokens=400
)
print(response.content[0].text)

18555375560


如果我们多次运行上述代码，可能会得到不同的答案，但这是 Claude 回复的一个答案：

```
18593367726060
```

实际正确的答案是：

```
18538003464660
```
Claude 偏差了 `55364261400`！

## 工具使用来救场了！

Claude 不擅长做复杂的数学运算，所以让我们通过提供一个计算器工具来增强 Claude 的能力。

下面是一个简单的图表来说明这个过程：



第一步是定义实际的计算器函数，并确保它在独立于 Claude 的情况下正常工作。我们将编写一个非常简单的函数，它接受三个参数：
* 一个操作，如 "add" 或 "multiply"
* 两个操作数

下面是一个基本实现：

In [23]:
def calculator(operation, operand1, operand2):
    if operation == "add":
        return operand1 + operand2
    elif operation == "subtract":
        return operand1 - operand2
    elif operation == "multiply":
        return operand1 * operand2
    elif operation == "divide":
        if operand2 == 0:
            raise ValueError("Cannot divide by zero.")
        return operand1 / operand2
    else:
        raise ValueError(f"Unsupported operation: {operation}")

请注意，这个简单的函数在实用性方面非常有限，因为它只能处理简单的表达式，如 `234 + 213` 或 `3 * 9`。这里的重点是通过一个非常简单的教育示例来了解使用工具的过程。

让我们测试一下我们的函数，确保它能正常工作。

In [24]:
calculator("add", 10, 3)

13

In [25]:
calculator("divide", 200, 25)

8.0

下一步是定义我们的工具并告诉 Claude 它的存在。在定义工具时，我们遵循一个非常特定的格式。每个工具定义包括：

* `name`：工具的名称。必须匹配正则表达式 ^[a-zA-Z0-9_-]{1,64}$。
* `description`：关于工具功能、使用时机和行为的详细纯文本描述。
* `input_schema`：定义工具预期参数的 JSON Schema 对象。

不熟悉 JSON Schema？[在此了解更多](https://json-schema.org/learn/getting-started-step-by-step)。

以下是一个假设工具的简单示例：

```json
{
  "name": "send_email",
  "description": "Sends an email to the specified recipient with the given subject and body.",
  "input_schema": {
    "type": "object",
    "properties": {
      "to": {
        "type": "string",
        "description": "The email address of the recipient"
      },
      "subject": {
        "type": "string",
        "description": "The subject line of the email"
      },
      "body": {
        "type": "string",
        "description": "The content of the email message"
      }
    },
    "required": ["to", "subject", "body"]
  }
}
```

这个名为 `send_email` 的工具期望以下输入：
* `to`，一个字符串，是必需的
* `subject`，一个字符串，是必需的
* `body`，一个字符串，是必需的


这是另一个名为 `search_product` 的工具定义：

```json
{
  "name": "search_product",
  "description": "Search for a product by name or keyword and return its current price and availability.",
  "input_schema": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "The product name or search keyword, e.g. 'iPhone 13 Pro' or 'wireless headphones'"
      },
      "category": {
        "type": "string",
        "enum": ["electronics", "clothing", "home", "toys", "sports"],
        "description": "The product category to narrow down the search results"
      },
      "max_price": {
        "type": "number",
        "description": "The maximum price of the product, used to filter the search results"
      }
    },
    "required": ["query"]
  }
}
```
这个工具包含3个输入：
* 一个必需的 `query` 字符串，表示产品名称或搜索关键词
* 一个可选的 `category` 字符串，必须是预定义值之一以缩小搜索范围。注意定义中的 `"enum"`。
* 一个可选的 `max_price` 数字，用于过滤低于某个价格点的结果

### 我们的计算器工具定义
让我们为我们之前编写的计算器函数定义相应的工具。我们知道计算器函数有3个必需参数：
* `operation` - 只能是 "add"、"subtract"、"multiply" 或 "divide"
* `operand1`，应该是一个数字
* `operand2`，也应该是一个数字

以下是工具定义：

In [ ]:
calculator_tool = {
    "name": "calculator",
    "description": "A simple calculator that performs basic arithmetic operations.",
    "input_schema": {
        "type": "object",
        "properties": {
            "operation": {
                "type": "string",
                "enum": ["add", "subtract", "multiply", "divide"],
                "description": "The arithmetic operation to perform."
            },
            "operand1": {
                "type": "number",
                "description": "The first operand."
            },
            "operand2": {
                "type": "number",
                "description": "The second operand."
            }
        },
        "required": ["operation", "operand1", "operand2"]
    }
}

***

## 练习

让我们练习使用以下函数作为示例来编写一个正确格式化的工具定义：

In [ ]:
def inventory_lookup(product_name, max_results):
    return "this function doesn't do anything"
    #You do not need to touch this or do anything with it!

这个假设的 `inventory_lookup` 函数应该这样调用：

In [ ]:
inventory_lookup("AA batteries", 4)

inventory_lookup("birthday candle", 10)

你的任务是编写一个相应的、格式正确的工具定义。在你的工具定义中，假设两个参数都是必需的。

***

### 向 Claude 提供我们的工具
现在回到我们之前的计算器函数。此时，Claude 对计算器工具一无所知！它只是一个小的 Python 字典。在向 Claude 发出请求时，我们可以传递一个工具列表来"告诉"Claude 它的存在。让我们现在就试试：

In [38]:
response = client.messages.create(
    model="claude-3-haiku-20240307",
    messages=[{"role": "user", "content": "Multiply 1984135 by 9343116. Only respond with the result"}],
    max_tokens=300,
    # Tell Claude about our tool
    tools=[calculator_tool]
)

接下来，让我们看看 Claude 给我们的回复：

In [42]:
response

ToolsBetaMessage(id='msg_01UfKwdmEsgTh99wfpgW4NJ7', content=[ToolUseBlock(id='toolu_015wQ7Wipo589yT9B3YTwjF1', input={'operand1': 1984135, 'operand2': 9343116, 'operation': 'multiply'}, name='calculator', type='tool_use')], model='claude-3-haiku-20240307', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(input_tokens=420, output_tokens=93))

```
ToolsBetaMessage(id='msg_01UfKwdmEsgTh99wfpgW4NJ7', content=[ToolUseBlock(id='toolu_015wQ7Wipo589yT9B3YTwjF1', input={'operand1': 1984135, 'operand2': 9343116, 'operation': 'multiply'}, name='calculator', type='tool_use')], model='claude-3-haiku-20240307', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(input_tokens=420, output_tokens=93))
```

你可能注意到我们的回复看起来与平时有些不同！具体来说，我们现在得到的不是普通的 `Message`，而是 `ToolsMessage`。

此外，我们可以检查 `response.stop_reason`，看到 Claude 停止是因为它决定是时候使用工具了：


In [44]:
response.stop_reason

'tool_use'

`response.content` 包含一个列表，其中有一个 `ToolUseBlock`，它本身包含有关工具名称和输入的信息：

In [43]:
response.content

[ToolUseBlock(id='toolu_015wQ7Wipo589yT9B3YTwjF1', input={'operand1': 1984135, 'operand2': 9343116, 'operation': 'multiply'}, name='calculator', type='tool_use')]

In [48]:
tool_name = response.content[0].name
tool_inputs = response.content[0].input

print("The Tool Name Claude Wants To Call:", tool_name)
print("The Inputs Claude Wants To Call It With:", tool_inputs)

The Tool Name Claude Wants To Call: calculator
The Inputs Claude Wants To Call It With: {'operand1': 1984135, 'operand2': 9343116, 'operation': 'multiply'}


下一步是简单地获取 Claude 提供给我们的工具名称和输入，并使用它们来实际调用我们之前编写的计算器函数。然后我们就会得到最终答案！

In [49]:
operation = tool_inputs["operation"]
operand1 = tool_inputs["operand1"]
operand2 = tool_inputs["operand2"]

result = calculator(operation, operand1, operand2)
print("RESULT IS", result)

RESULT IS 18538003464660


我们得到了正确答案 `18538003464660`！！！我们不是依赖 Claude 来正确计算，而是简单地询问 Claude 一个问题，并在必要时给它访问一个它可以决定使用的工具。

#### 重要提示
如果我们要问 Claude 一些不需要使用工具的问题，在这种情况下，即与数学或计算无关的问题，我们可能希望它正常回复。Claude 通常会这样做，但有时 Claude 非常热衷于使用它的工具！

这里有一个例子，Claude 有时会尝试使用计算器，尽管使用它没有意义。让我们看看当我们问 Claude "翡翠是什么颜色的？"时会发生什么。

In [77]:
response = client.messages.create(
    model="claude-3-haiku-20240307",
    messages=[{"role": "user", "content":"What color are emeralds?"}],
    max_tokens=400,
    tools=[calculator_tool]
)

In [78]:
response

ToolsBetaMessage(id='msg_01Dj82HdyrxGJpi8XVtqEYvs', content=[ToolUseBlock(id='toolu_01Xo7x3dV1FVoBSGntHNAX4Q', input={'operand1': 0, 'operand2': 0, 'operation': 'add'}, name='calculator', type='tool_use')], model='claude-3-haiku-20240307', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(input_tokens=409, output_tokens=89))

Claude 给我们这个回复：

```
ToolsBetaMessage(id='msg_01Dj82HdyrxGJpi8XVtqEYvs', content=[ToolUseBlock(id='toolu_01Xo7x3dV1FVoBSGntHNAX4Q', input={'operand1': 0, 'operand2': 0, 'operation': 'add'}, name='calculator', type='tool_use')], model='claude-3-haiku-20240307', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(input_tokens=409, output_tokens=89))

```
Claude 希望我们调用计算器工具？一个非常简单的修复方法是调整我们的提示或添加一个系统提示，内容类似于："你可以访问工具，但只在必要时使用它们。如果不需要工具，正常回复"：

In [79]:
response = client.messages.create(
    model="claude-3-haiku-20240307",
    system="You have access to tools, but only use them when necessary.  If a tool is not required, respond as normal",
    messages=[{"role": "user", "content":"What color are emeralds?"}],
    max_tokens=400,
    tools=[calculator_tool]
)

In [80]:
response

ToolsBetaMessage(id='msg_01YRRfnUUhP1u5ojr9iWZGGu', content=[TextBlock(text='Emeralds are green in color.', type='text')], model='claude-3-haiku-20240307', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(input_tokens=434, output_tokens=12))

现在 Claude 回复了适当的内容，不会牵强地在没有意义的时候使用工具。这是我们收到的新回复：

```
'Emeralds are green in color.'
```

我们还可以看到 `stop_reason` 现在是 `end_turn` 而不是 `tool_use`。

In [81]:
response.stop_reason

'end_turn'

***

### 把它们整合在一起

In [2]:
def calculator(operation, operand1, operand2):
    if operation == "add":
        return operand1 + operand2
    elif operation == "subtract":
        return operand1 - operand2
    elif operation == "multiply":
        return operand1 * operand2
    elif operation == "divide":
        if operand2 == 0:
            raise ValueError("Cannot divide by zero.")
        return operand1 / operand2
    else:
        raise ValueError(f"Unsupported operation: {operation}")


calculator_tool = {
    "name": "calculator",
    "description": "A simple calculator that performs basic arithmetic operations.",
    "input_schema": {
        "type": "object",
        "properties": {
            "operation": {
                "type": "string",
                "enum": ["add", "subtract", "multiply", "divide"],
                "description": "The arithmetic operation to perform.",
            },
            "operand1": {"type": "number", "description": "The first operand."},
            "operand2": {"type": "number", "description": "The second operand."},
        },
        "required": ["operation", "operand1", "operand2"],
    },
}


def prompt_claude(prompt):
    messages = [{"role": "user", "content": prompt}]
    response = client.messages.create(
        model="claude-3-haiku-20240307",
        system="You have access to tools, but only use them when necessary. If a tool is not required, respond as normal",
        messages=messages,
        max_tokens=500,
        tools=[calculator_tool],
    )

    if response.stop_reason == "tool_use":
        tool_use = response.content[-1]
        tool_name = tool_use.name
        tool_input = tool_use.input

        if tool_name == "calculator":
            print("Claude wants to use the calculator tool")
            operation = tool_input["operation"]
            operand1 = tool_input["operand1"]
            operand2 = tool_input["operand2"]

            try:
                result = calculator(operation, operand1, operand2)
                print("Calculation result is:", result)
            except ValueError as e:
                print(f"Error: {str(e)}")

    elif response.stop_reason == "end_turn":
        print("Claude didn't want to use a tool")
        print("Claude responded with:")
        print(response.content[0].text)


In [85]:
prompt_claude("I had 23 chickens but 2 flew away.  How many are left?")

Claude want to use the calculator tool
Calculation result is:  21


In [86]:
prompt_claude("What is 201 times 2")

Claude want to use the calculator tool
Calculation result is:  402


In [87]:
prompt_claude("Write me a haiku about the ocean")

Claude didn't want to use a tool
Claude responded with: 
Here is a haiku about the ocean:

Vast blue expanse shines,
Waves crash upon sandy shores,
Ocean's soothing song.


*** 

## 练习

你的任务是帮助构建一个使用 Claude 的研究助手。用户可以输入一个他们想要研究的主题，并获得一份 Wikipedia 文章链接列表，保存到 markdown 文件中供以后阅读。我们可以直接要求 Claude 生成文章 URL 列表，但 Claude 在 URL 方面不可靠，可能会产生虚构的文章 URL。此外，合法的文章可能在 Claude 的训练截止日期之后已经迁移到新的 URL。相反，我们将使用一个连接到真实 Wikipedia API 的工具来实现这一点！

我们将向 Claude 提供一个工具，该工具接受 Claude 生成的可能的 Wikipedia 文章标题列表（可能是虚构的）。我们可以使用这个工具在 Wikipedia 上搜索真实的 Wikipedia 文章标题和 URL，以确保最终的列表由实际存在的文章组成。然后我们会将这些文章 URL 保存到一个 markdown 文件中供以后阅读。

我们为你提供了两个辅助函数：

In [4]:
import wikipedia
def generate_wikipedia_reading_list(research_topic, article_titles):
    wikipedia_articles = []
    for t in article_titles:
        results = wikipedia.search(t)
        try:
            page = wikipedia.page(results[0])
            title = page.title
            url = page.url
            wikipedia_articles.append({"title": title, "url": url})
        except:
            continue
    add_to_research_reading_file(wikipedia_articles, research_topic)

def add_to_research_reading_file(articles, topic):
    with open("output/research_reading.md", "a", encoding="utf-8") as file:
        file.write(f"## {topic} \n")
        for article in articles:
            title = article["title"]
            url = article["url"]
            file.write(f"* [{title}]({url}) \n")
        file.write(f"\n\n")

第一个函数 `generate_wikipedia_reading_list` 期望接收一个研究主题，如"夏威夷的历史"或"世界各地的海盗"，以及一个可能的 Wikipedia 文章名称列表，我们将让 Claude 生成。这个函数使用 `wikipedia` 包来搜索对应的真实 wikipedia 页面，并构建一个包含文章标题和 URL 的字典列表。

然后它调用 `add_to_research_reading_file`，传入 Wikipedia 文章数据列表和总体研究主题。这个函数只是将每个 Wikipedia 文章的 markdown 链接添加到名为 `output/research_reading.md` 的文件中。文件名目前是硬编码的，函数假设它存在。它在这个仓库中存在，但如果在其他地方工作，你需要自己创建它。

我们的想法是让 Claude "调用" `generate_wikipedia_reading_list`，并传入一个可能的文章标题列表，其中一些可能是真实的，一些可能不是。Claude 可能会传入以下文章标题输入列表，其中一些是真实的 Wikipedia 文章，一些不是：

```py
["Piracy", "Famous Pirate Ships", "Golden Age Of Piracy", "List of Pirates", "Pirates and Parrots", "Piracy in the 21st Century"]
```

`generate_wikipedia_reading_list` 函数遍历每个文章标题，并收集任何实际存在的 Wikipedia 文章的真实文章标题和相应的 URL。然后它调用 `add_to_research_reading_file` 将这些内容写入一个 markdown 文件供以后参考。

### 最终目标

你的工作是实现一个名为 `get_research_help` 的函数，它接受一个研究主题和所需的文章数量。这个函数应该使用 Claude 实际生成可能的 Wikipedia 文章列表，并调用上面的 `generate_wikipedia_reading_list` 函数。以下是一些示例函数调用：

```py
get_research_help("Pirates Across The World", 7)

get_research_help("History of Hawaii", 3)

get_research_help("are animals conscious?", 3)
```

在这3次函数调用之后，这是我们的输出 `research_reading.md` 文件的样子（你自己可以在 output/research_reading.md 中查看）：





为了完成这项任务，你需要做以下事情：

* 为 `generate_wikipedia_reading_list` 函数编写一个工具定义
* 实现 `get_research_help` 函数
    * 写一个提示给 Claude，告诉它你需要帮助收集关于特定主题的研究，以及你想要它生成多少个文章标题
    * 告诉 Claude 它可以访问的工具
    * 向 Claude 发送你的请求
    * 检查 Claude 是否调用了工具。如果调用了，你需要将文章标题和它生成的主题传递给 `generate_wikipedia_reading_list` 函数。该函数将收集真实的 Wikipedia 文章链接，然后调用 `add_to_research_reading_file` 将链接写入 `output/research_reading.md`
    * 打开 `output/research_reading.md` 看看是否成功！


##### 起始代码

In [5]:
# Here's your starter code!
import wikipedia
def generate_wikipedia_reading_list(research_topic, article_titles):
    wikipedia_articles = []
    for t in article_titles:
        results = wikipedia.search(t)
        try:
            page = wikipedia.page(results[0])
            title = page.title
            url = page.url
            wikipedia_articles.append({"title": title, "url": url})
        except:
            continue
    add_to_research_reading_file(wikipedia_articles, research_topic)

def add_to_research_reading_file(articles, topic):
    with open("output/research_reading.md", "a", encoding="utf-8") as file:
        file.write(f"## {topic} \n")
        for article in articles:
            title = article["title"]
            url = article["url"]
            file.write(f"* [{title}]({url}) \n")
        file.write(f"\n\n")
        
def get_research_help(topic, num_articles=3):
   #Implement this function! 
   pass